# 03 — Stockout Math

Velocity vs on-hand. One formula runs the reorder sheet, urgency bands, and dead-stock flip side.

Outputs from demo (Tote 2 left, 2.3d) + corpus_big stockout/deadstock shops.

## 0. The three numbers

- `sold_7d` — units sold last 7 days (from orders, normalized product name)
- `on_hand` — inventory.csv `stock` (fuzzy product match, case-insensitive)
- `velocity = sold_7d / 7` (units/day)

In [1]:
from omni_one.packs.seller_os import ingest_seller_folder, build_seller_briefing
print('imports: stdlib + omni_one.packs.seller_os only')

imports: stdlib + omni_one.packs.seller_os only


## 1. Formula

```
daily_velocity = sold_7d / 7
days_left = on_hand / velocity  (999 if velocity==0)
reorder_qty = ceil(velocity * 14 - on_hand)  # 14-day cover
```

`COVER_DAYS=14`, `STOCKOUT_DAYS=5.0` in `seller_close.py` (matches `SCENARIO_THRESHOLDS`).

In [1]:
import math
def days_left(on_hand, sold_7d):
    vel=sold_7d/7 if sold_7d else 0
    return round(on_hand/vel,1) if vel else 999
def reorder_qty(on_hand, sold_7d, cover=14):
    vel=sold_7d/7 if sold_7d else 0
    return max(0, math.ceil(vel*cover - on_hand)) if vel else 0
print(days_left(2,6), reorder_qty(2,6))
print(days_left(15,3), reorder_qty(15,3))

2.3 10
35.0 0


## 2. Worked Tote example (demo golden)

Tote Bag Handmade: sold 6 in 7d, on-hand 2.

`vel=6/7=0.86/d`, `days=2/0.86=2.3d`, `need=ceil(0.86*14-2)=ceil(10.04)=10`. Citation `[inventory.csv:2]`.

In [1]:
events,_ = ingest_seller_folder('/tmp/seller_demo_check')
b = build_seller_briefing(events)
print(b['stockout_risk'])

[{'product': 'Tote Bag Handmade', 'on_hand': 2, 'sold_7d': 6, 'days_supply': 2.3, 'citation': '[inventory.csv:2] Tote Bag Handmade on_hand=2'}]


### Close reorder row

Adds `supplier` (from inventory `supplier` column), `urgency`, `citation`. Sorted by `days_left` ascending.

In [1]:
from omni_one.packs.seller_close import build_daily_close
from omni_one.packs.seller_scenarios import run_all_scenarios
scen = run_all_scenarios(events)
close = build_daily_close(events, b, scen, shop='demo')
print(close['reorder_sheet'][0])

{'product': 'Tote Bag Handmade', 'on_hand': 2, 'sold_7d': 6, 'daily_velocity': 0.86, 'days_left': 2.3, 'reorder_qty': 10, 'cover_days': 14, 'supplier': 'Acme Textiles', 'urgency': 'ORDER TODAY', 'citation': '[inventory.csv:2] Tote Bag Handmade on_hand=2'}


## 3. Urgency bands

| days_left | urgency | action |
|---|---|---|
| `<3` | `ORDER TODAY` | send PO now |
| `3–5` | `this week` | confirm ETA |
| `≥5` | `watch` (not in sheet) | monitor |

Demo Tote 2.3d → ORDER TODAY. Sheet only lists `<5d` (briefing `risk` filter).

In [1]:
for d in [1.2, 2.3, 4.1, 6.0, 35.0]:
    u='ORDER TODAY' if d<3 else ('this week' if d<5 else 'watch')
    print(f"{d}d -> {u}")

1.2d -> ORDER TODAY
2.3d -> ORDER TODAY
4.1d -> this week
6.0d -> watch
35.0d -> watch


## 4. Why 14-day cover?

7d velocity is noisy (weekend spike). 14d = 2× week, covers supplier lead time for micro-shops (Acme 5–7d + buffer). `ceil` avoids fractional units. Tune `COVER_DAYS` in `seller_close.py` if supplier is slower.

In [1]:
vel=0.86
for cover in [7,14,21,30]:
    import math
    print(f"cover {cover}d -> {math.ceil(vel*cover-2)} units")

cover 7d -> 5 units
cover 14d -> 10 units
cover 21d -> 17 units
cover 30d -> 24 units


## 5. Fuzzy product match

`Tote Bag Handmade` vs `tote bag` vs `Tote Bag Canvas` all match via `_norm_product` + `_product_match` (exact → substring → Jaccard → SequenceMatcher 0.72). Case-insensitive — old exact-match missed `Tote Bag` vs `tote bag`.

In [1]:
from omni_one.packs.seller_parse import _norm_product, _product_match
print(_norm_product('Tote Bag Handmade'))
print(_product_match('Tote Bag Handmade','tote bag'))
print(_product_match('Ceramic Mug','mug'))

tote bag handmade
True
True


## 6. Stockout shop (corpus_big shop_02)

Leather Wallet 17u sold, low stock → `STOCKOUT_RISK`. Archetype forces 75% orders = best seller, stock 1–2.

In [1]:
e2,_ = ingest_seller_folder('evals/corpus_big/shop_02')
b2 = build_seller_briefing(e2)
print(b2['kpis']['best_seller'])
print(b2['stockout_risk'])

{'product': 'Leather Wallet', 'qty': 17, 'gmv': 665.0}
[{'product': 'Leather Wallet', 'on_hand': 1, 'sold_7d': 17, 'days_supply': 0.4, 'citation': '[inventory.csv:2] Leather Wallet on_hand=1'}]


### Its reorder

`vel=17/7=2.43/d`, `days=1/2.43=0.4d`, `need=ceil(2.43*14-1)=33`. ORDER TODAY — more urgent than demo.

In [1]:
import math
vel=17/7
print(f"vel={vel:.2f} days={1/vel:.1f} need={math.ceil(vel*14-1)}")

vel=2.43 days=0.4 need=33


## 7. Dead-stock logic (flip side)

If `on_hand >=5` and **zero sales 7d** → `DEAD_STOCK`: bundle/clearance, don't reorder. Thresholds: `dead_stock_days=30`, `dead_stock_min_qty=5` (ctx window is 7d in evals).

In [1]:
e5,_ = ingest_seller_folder('evals/corpus_big/shop_05')
from omni_one.packs.seller_scenarios import run_all_scenarios as ras
s5 = ras(e5)
print(s5['by_scenario'])
print('dead-stock archetype: 2 lines 30+ units, zero sales -> bundle/clearance')

{'STOCKOUT_RISK': 0, 'DEAD_STOCK': 2, 'UNANSWERED_COMPLAINT': 2, 'REFUND_RISK': 1, 'REVIEW_CRISIS': 0, 'DM_BACKLOG': 1, 'FEE_CREEP': 0, 'SHIPPING_LOSS': 1, 'PROFIT_DIP': 0, 'SUPPLIER_COGS_SPIKE': 1, 'PRICE_MISMATCH': 0, 'LISTING_GAP': 1}
dead-stock archetype: 2 lines 30+ units, zero sales -> bundle/clearance


### Dead-stock note in close

`close['dead_stock_note']` lists up to 3 `DEAD_STOCK` decisions: title + evidence. Action: `markDownProduct → CLEARANCE`. Don't tie cash in slow SKUs.

In [1]:
c5 = build_daily_close(e5, build_seller_briefing(e5), s5, shop='shop_05')
print(c5['dead_stock_note'][:1])
print('cash locked -> clearance/bundle, free COGS for winners')

[{'title': 'Dead stock: Tote Bag Handmade \u2014 38 units, zero sales 7d \u2192 bundle/clearance', 'evidence': '[inventory.csv:5] Tote Bag Handmade on_hand=38'}]
cash locked -> clearance/bundle, free COGS for winners


## 8. PO draft

`draft_po_email(supplier, lines)`: lists top-5 reorders with `have X, Yd left` + citations. Demo PO → Acme Textiles ×10 Tote.

In [1]:
print(close['po_draft'][:300])

Subject: Reorder request — Acme Textiles

Hi Acme Textiles!

Please send the following for demo:
  - Tote Bag Handmade: 10 units (have 2, 2.3d left) [[inventory.csv:2] Tote Bag Handmade on_hand=2]


## 9. Checklist

- [ ] Velocity = sold/7, not sold/30 (responsive).
- [ ] Days <5 → sheet, <3 → today.
- [ ] Cover 14d, `ceil`, supplier from inventory.
- [ ] Zero-sales + ≥5 on-hand → dead-stock, not reorder.
- [ ] Every row cites `[file:line]`.

In [1]:
print('velocity=6/7=0.86 days=2.3 need=10 ORDER TODAY (demo Tote)')

velocity=6/7=0.86 days=2.3 need=10 ORDER TODAY (demo Tote)
